# Data preparation and evaluation

Turns the three training files into training data and defines the metric suite every later number
comes from. No model is involved.

Four preparation decisions, each justified by a measurement:
deduplicate, strip stereochemistry from the answers, write each input molecule several ways,
and split off a development set grouped by molecule.

In [1]:
import json, random, sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "utils"))

from src import evaluation as ev
import molecule_utils as mu          # the provided helpers

pd.set_option("display.width", 200, "display.max_columns", 20)

TRAIN_FILES = ["drugbank.csv", "gloryx.csv", "metxbiodb.csv"]
raw = pd.concat(
    [pd.read_csv(ROOT / "data" / f, dtype=str, keep_default_na=False, na_values=[]).assign(file=f)
     for f in TRAIN_FILES],
    ignore_index=True,
)
print(f"{len(raw):,} rows across {len(TRAIN_FILES)} files")
raw.groupby("file").size().to_frame("rows")

7,487 rows across 3 files


,rows
file,
drugbank.csv,3088
gloryx.csv,2364
metxbiodb.csv,2035


## 1. Deduplicate

The three sources overlap. The same parent-metabolite pair is often recorded in two or three of
them, so concatenating the files gives that chemistry two or three times the weight.

In [2]:
raw["parent"] = [ev.canonical(s) for s in raw["parent_smiles"]]
raw["metabolite"] = [ev.canonical(s) for s in raw["metabolite_smiles"]]

deduped = raw.drop_duplicates(subset=["parent", "metabolite"]).reset_index(drop=True)
print(f"  rows as concatenated          : {len(raw):,}")
print(f"  distinct (parent, metabolite) : {len(deduped):,}")
print(f"  cross-file duplicates removed : {len(raw) - len(deduped):,} "
      f"({(len(raw) - len(deduped)) / len(raw):.1%})")

  rows as concatenated          : 7,487
  distinct (parent, metabolite) : 5,679
  cross-file duplicates removed : 1,808 (24.1%)


A quarter of the apparent data was the same chemistry counted twice. Any statement of training
size has to quote the smaller number.

## 2. Remove chemistry that has no analogue in the holdout test set

Two groups are dropped, for different reasons.

**Parents with no carbon atoms.** Iron three becoming iron two, hydrogen peroxide becoming oxygen,
nitrate, nitric oxide, arsenite, selenite, borate. This is not drug metabolism, and training on it
teaches that swapping a single atom is a plausible metabolic edit. The decision does not depend on
the test set.

**Parents over 100 heavy atoms.** Each loses between 6 and 158 carbons, which is proteolysis or
glycoside cleavage. **None of those transformations occurs in the holdout test set**, where the
dominant reactions are hydroxylation at 31%, demethylation at 11% and glucuronidation at 10%. Four
molecules cannot teach that chemistry, and at 226 to 647 characters against a training median of
41 they consume eleven times their share of input length.

This second group is a **scoping decision**. The model is trained on
molecules of roughly 8 to 62 heavy atoms and should not be expected to generalise to peptide
drugs. That class needs a dedicated dataset; GOSTAR has only 18 large amide-rich molecules out of
11,026, so no provided file covers it.

The holdout test set does contain peptide-*like* drugs, seven molecules with three or more amide
bonds, up to 47 atoms. Those stay: only the size class is missing from the test set.

In [3]:
from rdkit import Chem

def n_carbons(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "C") if mol else 0

def n_heavy(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumHeavyAtoms() if mol else 0

inorganic = deduped["parent"].map(lambda s: n_carbons(s) == 0)
oversized = deduped["parent"].map(lambda s: n_heavy(s) > 100)
drop = inorganic | oversized

print(f"  inorganic parents (no carbon) : {inorganic.sum():>3} pairs, "
      f"{deduped.loc[inorganic, 'parent'].nunique()} distinct molecules")
print(f"  parents over 100 heavy atoms  : {oversized.sum():>3} pairs, "
      f"{deduped.loc[oversized, 'parent'].nunique()} distinct molecules")
print(f"  removed in total              : {drop.sum():>3} pairs ({drop.mean():.2%})")

# the test set is never filtered; confirm it would be untouched anyway
test_check = pd.read_csv(ROOT / "data" / "test_set.csv", dtype=str, keep_default_na=False, na_values=[])
tp_chk = {ev.canonical(s) for s in test_check["parent_smiles"]}
tm_chk = {ev.canonical(s) for s in test_check["metabolite_smiles"]}
print()
print(f"  test parents this would have caught     : "
      f"{sum(1 for s in tp_chk if n_carbons(s) == 0 or n_heavy(s) > 100)} of {len(tp_chk)}")
print(f"  test metabolites this would have caught : "
      f"{sum(1 for s in tm_chk if n_carbons(s) == 0 or n_heavy(s) > 100)} of {len(tm_chk)}")

deduped = deduped[~drop].reset_index(drop=True)
print(f"\n  training pairs remaining : {len(deduped):,}")

  inorganic parents (no carbon) :  20 pairs, 11 distinct molecules
  parents over 100 heavy atoms  :   7 pairs, 4 distinct molecules
  removed in total              :  27 pairs (0.48%)

  test parents this would have caught     : 0 of 155
  test metabolites this would have caught : 0 of 302

  training pairs remaining : 5,652


## 3. Strip stereochemistry from the answers

Which stereoisomer a reaction produces depends on the shape of the enzyme's active site, and the
enzyme is not in the input, so the target depends on something the input cannot determine.

The data shows this is noise: the same skeleton appears in the training set
with more than one stereochemistry. Stripping it from the targets removes the contradiction and
costs almost nothing on the holdout test set, where it merges only 5 of 309 pairs.

Inputs keep their stereochemistry, because the test molecules have it.

In [4]:
deduped["target"] = [ev.canonical(s, keep_stereo=False) for s in deduped["metabolite"]]

flat = deduped.drop_duplicates(subset=["parent", "target"]).reset_index(drop=True)

# After standardising these read as "predict X from X". The holdout test set contains none.
echoes = [ev.standardize(p) == ev.standardize(t) for p, t in zip(flat["parent"], flat["target"])]
print(f"  pairs whose answer standardises to the input: {sum(echoes)} (removed)")
flat = flat[~pd.Series(echoes, index=flat.index)].reset_index(drop=True)
print(f"  pairs remaining: {len(flat):,}")
collapsed = len(deduped) - len(flat)
print(f"  pairs before stripping : {len(deduped):,}")
print(f"  pairs after stripping  : {len(flat):,}")
print(f"  collapsed into a twin  : {collapsed:,} ({collapsed / len(deduped):.1%})")
print()
print(f"  unique parent molecules    : {flat['parent'].nunique():,}")
print(f"  mean metabolites per parent: {len(flat) / flat['parent'].nunique():.2f}")


  pairs whose answer standardises to the input: 16 (removed)
  pairs remaining: 4,713
  pairs before stripping : 5,652
  pairs after stripping  : 4,713
  collapsed into a twin  : 939 (16.6%)

  unique parent molecules    : 2,562
  mean metabolites per parent: 1.84


The 939 collapsed pairs (16.6% of the pairs before stripping) were the same training example
written twice, which would reintroduce the weighting problem the first deduplication removed.

Stripping stereochemistry creates new duplicates, so a second pass is needed for an accurate pair
count. Quoting the pre-strip count as the training size would count that 16.6% twice.

## 4. Split off a development set, grouped by molecule

The learning rate, the number of samples to draw and the decoding settings are chosen on this
split.

The split is by **molecule**. A parent with four metabolites sends all four to the same
side. A row split would put three in training and one in dev, and the dev score would measure
memorisation.

Grouping on the raw SMILES string is not enough. The same molecule appears written different ways,
as an amide in one record and its imidic-acid tautomer in another, or with a stereocentre drawn the
other way round, and string grouping sends those to opposite sides. On this data that put **43
molecules on both sides, affecting 99 of 486 dev pairs, a fifth of the development set**. Each
would have been scored as generalisation while being memorisation.

The split therefore groups on the **standardised** form, the key the evaluation harness uses to
decide whether two molecules are the same answer.

In [5]:
flat["parent_key"] = [ev.standardize(s) for s in flat["parent"]]

rng = np.random.default_rng(0)
groups = np.array(sorted(flat["parent_key"].dropna().unique()))
rng.shuffle(groups)

n_dev = int(round(0.10 * len(groups)))
dev_groups = set(groups[:n_dev])

dev = flat[flat["parent_key"].isin(dev_groups)].reset_index(drop=True)
train = flat[~flat["parent_key"].isin(dev_groups)].reset_index(drop=True)

test = pd.read_csv(ROOT / "data" / "test_set.csv", dtype=str, keep_default_na=False, na_values=[])
test_keys = {ev.standardize(s) for s in test["parent_smiles"]}

print(f"  distinct SMILES strings for parents   : {flat['parent'].nunique():,}")
print(f"  distinct MOLECULES after standardising: {flat['parent_key'].nunique():,}"
      f"   ({flat['parent'].nunique() - flat['parent_key'].nunique()} were the same molecule written twice)")
print()
print(f"  train : {len(train):>6,} pairs over {train['parent_key'].nunique():>5,} molecules")
print(f"  dev   : {len(dev):>6,} pairs over {dev['parent_key'].nunique():>5,} molecules")
test_strings = {ev.canonical(s) for s in test["parent_smiles"]}
print(f"  test  : {len(test):>6,} pairs over {len(test_strings):>5,} molecules   (provided, untouched)")
print(f"          {len(test_strings)} distinct SMILES, {len(test_keys)} distinct molecules once standardised")
print()

overlap = set(train["parent_key"]) & set(dev["parent_key"])
print(f"  molecules in both train and dev : {len(overlap)}  (must be 0)")
assert not overlap, "molecule-grouped split is broken"

shared_with_test = test_keys & (set(train["parent_key"]) | set(dev["parent_key"]))
print(f"  molecules shared with the test  : {len(shared_with_test)}  (must be 0)")
assert not shared_with_test, "a training molecule appears in the test set"

# What grouping on raw SMILES strings would have cost: the naive split, run for real.
naive_rng = np.random.default_rng(0)
strings = np.array(sorted(flat["parent"].unique()))
naive_rng.shuffle(strings)
naive_dev_strings = set(strings[: int(round(0.10 * len(strings)))])

naive_dev = flat[flat["parent"].isin(naive_dev_strings)]
naive_train = flat[~flat["parent"].isin(naive_dev_strings)]
leaked = set(naive_train["parent_key"]) & set(naive_dev["parent_key"])
leaked_pairs = int(naive_dev["parent_key"].isin(leaked).sum())
print()
print(f"  grouping on the raw SMILES string instead would have put {len(leaked)} molecules on both")
print(f"  sides, affecting {leaked_pairs} of {len(naive_dev)} dev pairs "
      f"({leaked_pairs / max(len(naive_dev), 1):.0%} of the split)")


  distinct SMILES strings for parents   : 2,562
  distinct MOLECULES after standardising: 2,297   (265 were the same molecule written twice)

  train :  4,276 pairs over 2,067 molecules
  dev   :    437 pairs over   230 molecules
  test  :    309 pairs over   155 molecules   (provided, untouched)
          155 distinct SMILES, 154 distinct molecules once standardised

  molecules in both train and dev : 0  (must be 0)
  molecules shared with the test  : 0  (must be 0)

  grouping on the raw SMILES string instead would have put 43 molecules on both
  sides, affecting 99 of 486 dev pairs (20% of the split)


## 5. Write each input molecule several ways

A molecule has many valid SMILES strings. Writing each parent several ways teaches the model the
molecule itself.

Varying the **inputs** moved recall from 0.37 to 0.43 in published work on these same datasets.
Varying the **targets** collapses top-1 accuracy from 42.2% to 26.8%, so answers stay in one
canonical form and only inputs vary.

Variants come from `get_smiles_variants` in the provided helpers, which keeps the canonical
spelling among those it returns.

In [6]:
# Seeded: randomize_smiles uses the stdlib RNG, so the example count varies without one.
random.seed(0)

N_SPELLINGS = 4
# The opening <metabolite> tag goes in the PROMPT, so the model writes only the molecule.
PROMPT = "<parent>{}</parent>\n<metabolite>"
COMPLETION = "{}</metabolite>"

def build_examples(df, n_spellings):
    rows = []
    variants = {}
    for parent, target in zip(df["parent"], df["target"]):
        if parent not in variants:
            got = mu.get_smiles_variants(parent, n=n_spellings)
            variants[parent] = got or [parent]
        for i, spelling in enumerate(variants[parent]):
            rows.append({
                "parent": parent,
                "parent_spelling": spelling,
                "target": target,
                "prompt": PROMPT.format(spelling),
                "completion": COMPLETION.format(target),
                "is_canonical": spelling == parent,
            })
    return pd.DataFrame(rows)

train_examples = build_examples(train, N_SPELLINGS)
dev_examples = build_examples(dev, 1)      # dev is held out for evaluation

print(f"  training pairs            : {len(train):,}")
print(f"  training examples written : {len(train_examples):,} "
      f"({len(train_examples) / len(train):.1f} spellings per pair on average)")
print(f"  canonical spelling kept   : {train_examples['is_canonical'].sum():,} examples")
print()
# One PAIR per row: selecting on the parent alone would repeat its spellings per answer.
sizes = train_examples.groupby(["parent", "target"]).size()
readable = [pair for pair, n in sizes.items() if n == N_SPELLINGS and 25 <= len(pair[0]) <= 40]
parent, target = readable[0]
one_pair = train_examples[(train_examples["parent"] == parent) & (train_examples["target"] == target)]

print(f"  one pair, its input written {len(one_pair)} ways:")
for _, r in one_pair.iterrows():
    print(f"    {r['prompt']}" + ("  <- canonical" if r["is_canonical"] else ""))
print(f"    the answer never varies: {one_pair['completion'].iloc[0]}")


  training pairs            : 4,276
  training examples written : 17,075 (4.0 spellings per pair on average)
  canonical spelling kept   : 4,276 examples

  one pair, its input written 4 ways:
    <parent>C1CC2C3CCC(=O)C=C3CCC2C2CCC(C#C)(O)C12CC</parent>
<metabolite>
    <parent>O=C1CCC2C(=C1)CCC1C3C(CC)(CCC12)C(C#C)(O)CC3</parent>
<metabolite>
    <parent>C1C2(CC)C(C#C)(O)CCC2C2CCC3=CC(=O)CCC3C2C1</parent>
<metabolite>
    <parent>C#CC1(O)CCC2C3CCC4=CC(=O)CCC4C3CCC21CC</parent>
<metabolite>  <- canonical
    the answer never varies: C#CC1(OC2OC(C(=O)O)C(O)C(O)C2O)CCC2C3CCC4=CC(=O)CCC4C3CCC21CC</metabolite>


In [7]:
out = ROOT / "data" / "processed"
out.mkdir(exist_ok=True)

for name, df in [("train", train_examples), ("dev", dev_examples)]:
    path = out / f"{name}.jsonl"
    with path.open("w") as fh:
        for r in df.to_dict("records"):
            fh.write(json.dumps(r) + "\n")
    print(f"  wrote {path.relative_to(ROOT)}  ({len(df):,} examples, {path.stat().st_size / 1024:.0f} KB)")

pd.DataFrame([
    {"split": "train", "pairs": len(train), "molecules": train["parent_key"].nunique(),
     "examples written": len(train_examples)},
    {"split": "dev", "pairs": len(dev), "molecules": dev["parent_key"].nunique(),
     "examples written": len(dev_examples)},
    {"split": "test (provided)", "pairs": len(test), "molecules": len(test_keys),
     "examples written": 0},
]).set_index("split")

  wrote data/processed/train.jsonl  (17,075 examples, 6324 KB)
  wrote data/processed/dev.jsonl  (437 examples, 166 KB)


,pairs,molecules,examples written
split,,,
train,4276,2067,17075
dev,437,230,437
test (provided),309,154,0


## 6. The evaluation harness

`src/evaluation.py` produces every reported number, so model results and baselines share one code
path.

It scores under three definitions of when two molecules are the same answer. The gaps between them
separate chemistry errors from formatting errors.

In [8]:
test["parent"] = [ev.canonical(s) for s in test["parent_smiles"]]
test["metabolite"] = [ev.canonical(s) for s in test["metabolite_smiles"]]
tp, tr_ = list(test["parent"]), list(test["metabolite"])

print("baseline: predict the parent molecule unchanged\n")
ev.evaluate_all_criteria(tp, tr_, ev.copy_parent(tp), k_values=(1, 5)).round(3)

baseline: predict the parent molecule unchanged



recall_pair  ceiling  pct_of_ceiling  recall_parent  parent_hit_rate  precision_known  tanimoto_wrong  parent_echo_rate
criterion    k                                                                                                                         
exact        1          0.0    0.502             0.0            0.0              0.0              0.0           0.634               1.0
             5          0.0    0.958             0.0            0.0              0.0              0.0           0.634               1.0
nostereo     1          0.0    0.515             0.0            0.0              0.0              0.0           0.634               1.0
             5          0.0    0.964             0.0            0.0              0.0              0.0           0.634               1.0
standardized 1          0.0    0.515             0.0            0.0              0.0              0.0           0.634               1.0
             5          0.0    0.964             0.0            0.0              0.0              0.0           0.634               1.0

Zero correct under every definition, yet a similarity score of 0.63. Any similarity figure has to
be read against that floor, which is why the harness reports similarity only for predictions it
scored wrong.

`parent_echo_rate` of 1.0 shows this baseline handing the input back. The column exists because
similarity cannot detect that, as measured in the docstring of `src/decoding.py`.

In [9]:
train_parents = list(raw["parent"])
train_metabolites = list(raw["metabolite"])
nn = ev.nearest_neighbour(tp, train_parents, train_metabolites, k=10)

print("baseline: copy the metabolites of the most similar training molecule\n")
ev.evaluate(tp, tr_, nn, k_values=(1, 5, 10), criterion="standardized").round(3)

baseline: copy the metabolites of the most similar training molecule



,recall_pair,ceiling,pct_of_ceiling,recall_parent,parent_hit_rate,precision_known,tanimoto_wrong,parent_echo_rate
k,,,,,,,,
1,0.016,0.515,0.031,0.024,0.032,0.032,0.331,0.090
5,0.055,0.964,0.057,0.064,0.097,0.022,0.411,0.252
10,0.065,1.000,0.065,0.076,0.110,0.016,0.418,0.277


Retrieval reaches about 6% at k=10, so a model scoring well above that is doing more than looking
up a near neighbour.

It also returns the parent itself for a quarter of molecules at k=5, which no similarity figure
would reveal.

## 7. The harness checks itself

A ceiling that counts distinct answers rather than rows is correct at k=1 and wrong elsewhere, so
a check against a remembered number such as 0.502 passes while the function is broken.

The invariant that catches this class of error: a model that already knows every answer cannot
beat the ceiling. That holds for any test set and any matching rule.

In [10]:
violations = ev.check_ceiling_is_an_upper_bound(tp, tr_)
print(f"  violations: {len(violations)}")
for v in violations:
    print(f"    {v}")
assert not violations, "ceiling is not an upper bound"

print()
print("  ceilings, which depend on the matching rule:")
pd.DataFrame(
    {c: {k: ev.recall_ceiling(tp, tr_, k, c) for k in (1, 3, 5, 10)} for c in ev.CRITERIA}
).rename_axis("k").round(3)

  violations: 0

  ceilings, which depend on the matching rule:


,exact,nostereo,standardized
k,,,
1,0.502,0.515,0.515
3,0.851,0.861,0.864
5,0.958,0.964,0.964
10,1.000,1.000,1.000


## Dev scores versus holdout test scores

Dev scores are expected to come out above test scores. Dev is carved from DrugBank, GLORYx and
MetXBioDB while the holdout test set is curated differently, and close analogues of training
molecules survive a split that only separates exact molecules. The gap is the cost of selecting on
a proxy.